# Lesson 6: Multimodal Input

See [docs/06-multimodal.md](../docs/06-multimodal.md).

Requires a multimodal model (e.g. gpt-4o) and a publicly accessible image URL.

In [ ]:
import sys
from functools import partial
from typing import Optional
sys.path.insert(0, '..')

from src.init_env import set_environment_variables
from src.data_loader import load_mails, split_dataset, build_option_lists
from src.orchestration_helper import get_orchestration_deployment_url
from gen_ai_hub.proxy import get_proxy_client
from gen_ai_hub.orchestration.models.config import OrchestrationConfig
from gen_ai_hub.orchestration.models.llm import LLM
from gen_ai_hub.orchestration.models.message import UserMessage, TextContent, ImageUrlContent
from gen_ai_hub.orchestration.models.template import Template, TemplateValue
from gen_ai_hub.orchestration.service import OrchestrationService

set_environment_variables()
proxy_client = get_proxy_client()
api_url = get_orchestration_deployment_url(proxy_client.ai_core_client)
service = OrchestrationService(api_url=api_url, proxy_client=proxy_client)

mails = load_mails()
dev_set, _, _ = split_dataset(mails)
_, _, _, option_lists = build_option_lists(mails)
mail = dev_set[10]

In [ ]:
def send_multimodal_request(prompt: str, image_url: Optional[str] = None, _model: str = 'gpt-4o', **kwargs) -> str:
    content_parts = []
    if image_url:
        content_parts.append(ImageUrlContent(url=image_url))
    content_parts.append(TextContent(text=prompt.format(**kwargs) if kwargs else prompt))
    config = OrchestrationConfig(
        llm=LLM(name=_model),
        template=Template(messages=[UserMessage(content=content_parts)]),
    )
    answer = service.run(config=config)
    return answer.module_results.llm.choices[0].message.content

# Replace with a real image URL accessible by the model
example_image_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png'

prompt_text = '''Classify the message and image. Return compact JSON with keys urgency, sentiment, categories.
Message: {input}'''

result = send_multimodal_request(prompt_text, image_url=example_image_url, input=mail['message'])
print(result)